# Laboratorio de Random Forest con PySpark y BigQuery opcional

Ejemplo didáctico con datos sintéticos por defecto. Las métricas se calculan al ejecutar el cuaderno; no acreditan eficacia sobre fraude real ni un despliegue productivo.

Descarga `train_rf_bigquery.py` desde el mismo panel y colócalo junto a este notebook. El código compartido evita que el cuaderno y el script diverjan. Requiere Java 17 y Python 3.10–3.12 para el entorno Spark 3.5 de este ejemplo. La lectura/escritura BigQuery requiere conector, credenciales y permisos configurados por el usuario, y puede generar costes.


## 1. Dependencias del laboratorio

In [ ]:
%pip install "pyspark[ml,sql]==3.5.7" "numpy<2"


## 2. Modo de ejecución

`--demo` genera datos sintéticos en Spark local y rechaza destinos remotos. Para usar BigQuery, sustituye la lista por parámetros explícitos, por ejemplo `['--project_id', 'MI_PROYECTO', '--dataset', 'MI_DATASET', '--input_table', 'MI_TABLA']`. Un error de lectura se comunica y detiene el flujo; no activa datos ficticios.

La escritura está desactivada por defecto. `--output_table` solicita una tabla nueva y `--model_output` una ruta nueva; ninguno sobrescribe datos existentes.


In [ ]:
from train_rf_bigquery import (
    parse_args, init_spark, synthetic_transactions, load_data,
    build_and_train_pipeline, scored_rows,
)

CLI_ARGS = ["--demo"]
args = parse_args(CLI_ARGS)
spark = init_spark("RandomForest_Laboratorio", args.gcs_bucket, args.dataset, demo=args.demo)


## 3. Datos y validación

In [ ]:
df_raw = synthetic_transactions(spark) if args.demo else load_data(
    spark, f"{args.project_id}.{args.dataset}.{args.input_table}"
)
df_raw.printSchema()
df_raw.groupBy("is_fraud").count().show()


## 4. Entrenamiento y evaluación

La división train/test ocurre antes del cálculo de pesos y del ajuste de características. `CrossValidator` recibe el `Pipeline` completo: el indexado, la codificación y la imputación se ajustan dentro de cada fold. El conjunto de test permanece reservado para la evaluación final. El ejemplo valida el esquema y exige las dos clases; una muestra insuficiente detiene el entrenamiento.


In [ ]:
model, predictions = build_and_train_pipeline(df_raw)
scored = scored_rows(predictions)
scored.show(5)


## 5. Importancia de características del modelo calculado

In [ ]:
attributes = predictions.schema["features"].metadata["ml_attr"]["attrs"]
ordered = sorted(
    (attribute for group in attributes.values() for attribute in group),
    key=lambda attribute: attribute["idx"],
)
importances = model.stages[-1].featureImportances.toArray()
ranking = sorted(
    ((attribute["name"], float(importances[attribute["idx"]])) for attribute in ordered),
    key=lambda item: item[1], reverse=True,
)
for name, importance in ranking[:10]:
    print(f"{name}: {importance:.4f}")


## 6. Persistencia opcional

Estas operaciones solo se ejecutan si se han añadido sus argumentos. Se guarda el `PipelineModel` completo para reutilizar las mismas transformaciones. Los errores de guardado se propagan; la salida a BigQuery usa `errorifexists`.


In [ ]:
if args.model_output:
    model.write().save(args.model_output)
if args.output_table:
    output_table = f"{args.project_id}.{args.dataset}.{args.output_table}"
    scored.write.format("bigquery").option("table", output_table).mode("errorifexists").save()


## 7. Cierre y límites

El rendimiento y las métricas dependen de los datos, la memoria, las particiones y el clúster. La demo no acredita millones de filas, ausencia de OOM ni integración con un dashboard. Antes de un uso real se requieren validación temporal, evaluación de sesgo, calibración de umbral y revisión de costes y permisos.


In [ ]:
spark.stop()
